# Databricks notebook source

# 01 - Ingesting Lading

Este notebook cria a Lading Zone dentro de um Volume Databricks, facilitanto a comunicação e tranformação do dado.

## Objetivos

- Baixa e importa os arquivos Parquet do endpoint publico para o volume Databricks.
- Cria o particionamento de Tabelas por mês de extração.
- Aplicar testes técnicos mínimos.
- Adicionar metadados de origem e processamento.
- cria o Delta no Unity Catalog para rastreabilidade.

#### 00. Teste de VM Databricks

In [0]:
print("Databricks notebook is running")             
print(spark.version)

Databricks notebook is running
4.1.0


In [0]:
dbutils.fs.ls("dbfs:/")

[FileInfo(path='dbfs:/Volumes/', name='Volumes/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/Workspace/', name='Workspace/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-datasets/', name='databricks-datasets/', size=0, modificationTime=0)]

In [0]:
CATALOG_NAME = "workspace"
SCHEMA_NAME = "ifood_case"
VOLUME_NAME = "case_files"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")

display(
    spark.sql(
        f"DESCRIBE VOLUME {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}"
    )
)

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
case_files,workspace,ifood_case,lucas12345543@gmail.com,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


#### 01. Bibliotecas Usadas e Project Root

In [0]:
import os
import sys
import importlib
import urllib.request
from datetime import datetime, timezone
from typing import Dict, List

In [0]:
current_path = os.getcwd()

if current_path.endswith("/notebooks"):
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = current_path

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for module_name in ["src.ingest_landing", "src"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

import src.ingest_landing as landing

print("BASE_URL:", landing.BASE_URL)
print("DBFS LANDING:", landing.LANDING_DBFS_BASE_PATH)
print("LOCAL LANDING:", landing.LANDING_LOCAL_BASE_PATH)

from src.ingest_landing import ingest_landing, ensure_unity_catalog_volume

BASE_URL: https://d37ci6vzurychx.cloudfront.net/trip-data
DBFS LANDING: dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi
LOCAL LANDING: /Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi


#### 02. Passagem de parâmetros para extração e execução do código ingest_landing.py

In [0]:
YEAR = 2023
MONTHS = [1, 2, 3, 4, 5]
OVERWRITE = True

results = ingest_landing(
    dbutils=dbutils,
    year=YEAR,
    months=MONTHS,
    overwrite=OVERWRITE,
)

for result in results:
    print(result)

{'year': 2023, 'month': 1, 'source_url': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet', 'landing_path': 'dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet', 'status': 'success', 'file_size_bytes': 47673370, 'ingestion_started_at': '2026-05-21T23:19:28.973382+00:00', 'ingestion_finished_at': '2026-05-21T23:19:31.251700+00:00'}
{'year': 2023, 'month': 2, 'source_url': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet', 'landing_path': 'dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=02/yellow_tripdata_2023-02.parquet', 'status': 'success', 'file_size_bytes': 47748012, 'ingestion_started_at': '2026-05-21T23:19:31.251744+00:00', 'ingestion_finished_at': '2026-05-21T23:19:32.456052+00:00'}
{'year': 2023, 'month': 3, 'source_url': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet'

#### 03. Validação de Extração com PySpark

In [0]:
import src.ingest_landing as landing

print(landing.build_tlc_url(2023, 1))
print(landing.build_landing_path(2023, 1))

https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet


In [0]:
for month in ["01", "02", "03", "04", "05"]:
    path = (
        "dbfs:/Volumes/workspace/ifood_case/case_files/"
        f"landing/nyc_tlc/yellow_taxi/year=2023/month={month}"
    )
    print(path)
    display(dbutils.fs.ls(path))

dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01


path,name,size,modificationTime
dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,yellow_tripdata_2023-01.parquet,47673370,1779405571000


dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=02


path,name,size,modificationTime
dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=02/yellow_tripdata_2023-02.parquet,yellow_tripdata_2023-02.parquet,47748012,1779405572000


dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=03


path,name,size,modificationTime
dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=03/yellow_tripdata_2023-03.parquet,yellow_tripdata_2023-03.parquet,56127762,1779405574000


dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=04


path,name,size,modificationTime
dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=04/yellow_tripdata_2023-04.parquet,yellow_tripdata_2023-04.parquet,54222699,1779405575000


dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=05


path,name,size,modificationTime
dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=05/yellow_tripdata_2023-05.parquet,yellow_tripdata_2023-05.parquet,58654627,1779405576000


In [0]:
from functools import reduce
from pyspark.sql import functions as F

YEAR = 2023
MONTHS = ["01", "02", "03", "04", "05"]

def read_landing_month(year: int, month: str):
    path = (
        "dbfs:/Volumes/workspace/ifood_case/case_files/"
        f"landing/nyc_tlc/yellow_taxi/year={year}/month={month}/"
        f"yellow_tripdata_{year}-{month}.parquet"
    )

    df = spark.read.parquet(path)

    return (
        df.select(
            F.col("VendorID").cast("int").alias("VendorID"),
            F.col("passenger_count").cast("double").alias("passenger_count"),
            F.col("total_amount").cast("double").alias("total_amount"),
            F.col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
            F.col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
        )
        .withColumn("source_year", F.lit(year))
        .withColumn("source_month", F.lit(int(month)))
        .withColumn("source_file", F.lit(path))
    )

monthly_dfs = [
    read_landing_month(YEAR, month)
    for month in MONTHS
]

raw_required_df = reduce(
    lambda left, right: left.unionByName(right),
    monthly_dfs
)

print(f"Total records: {raw_required_df.count()}")

raw_required_df.printSchema()

display(raw_required_df.limit(10))

Total records: 16186386
root
 |-- VendorID: integer (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- source_year: integer (nullable = false)
 |-- source_month: integer (nullable = false)
 |-- source_file: string (nullable = false)



VendorID,passenger_count,total_amount,tpep_pickup_datetime,tpep_dropoff_datetime,source_year,source_month,source_file
2,1.0,14.3,2023-01-01T00:32:10.000Z,2023-01-01T00:40:36.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,16.9,2023-01-01T00:55:08.000Z,2023-01-01T01:01:27.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,34.9,2023-01-01T00:25:04.000Z,2023-01-01T00:37:49.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
1,0.0,20.85,2023-01-01T00:03:48.000Z,2023-01-01T00:13:25.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,19.68,2023-01-01T00:10:29.000Z,2023-01-01T00:21:19.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,27.8,2023-01-01T00:50:34.000Z,2023-01-01T01:02:52.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,20.52,2023-01-01T00:09:22.000Z,2023-01-01T00:19:49.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,64.44,2023-01-01T00:27:12.000Z,2023-01-01T00:49:56.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,28.38,2023-01-01T00:21:44.000Z,2023-01-01T00:36:40.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet
2,1.0,19.9,2023-01-01T00:39:42.000Z,2023-01-01T00:50:36.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet


In [0]:
display(
    raw_required_df
    .groupBy("source_year", "source_month")
    .count()
    .orderBy("source_year", "source_month")
)

source_year,source_month,count
2023,1,3066766
2023,2,2913955
2023,3,3403766
2023,4,3288250
2023,5,3513649


#### 04. Salva manifesto de ingestão para controle de carga

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    LongType,
)

manifest_schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("source_url", StringType(), True),
    StructField("landing_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("file_size_bytes", LongType(), True),
    StructField("ingestion_started_at", StringType(), True),
    StructField("ingestion_finished_at", StringType(), True),
])

manifest_df = spark.createDataFrame(results, schema=manifest_schema)

manifest_path = (
    "dbfs:/Volumes/workspace/ifood_case/case_files/"
    "metadata/landing_ingestion_manifest"
)

(
    manifest_df
    .write
    .mode("append")
    .format("parquet")
    .save(manifest_path)
)

display(manifest_df)

year,month,source_url,landing_path,status,file_size_bytes,ingestion_started_at,ingestion_finished_at
2023,1,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,success,47673370,2026-05-21T23:19:28.973382+00:00,2026-05-21T23:19:31.251700+00:00
2023,2,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=02/yellow_tripdata_2023-02.parquet,success,47748012,2026-05-21T23:19:31.251744+00:00,2026-05-21T23:19:32.456052+00:00
2023,3,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=03/yellow_tripdata_2023-03.parquet,success,56127762,2026-05-21T23:19:32.456095+00:00,2026-05-21T23:19:33.770442+00:00
2023,4,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-04.parquet,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=04/yellow_tripdata_2023-04.parquet,success,54222699,2026-05-21T23:19:33.770479+00:00,2026-05-21T23:19:35.000745+00:00
2023,5,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-05.parquet,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=05/yellow_tripdata_2023-05.parquet,success,58654627,2026-05-21T23:19:35.000782+00:00,2026-05-21T23:19:36.440007+00:00
